# 🚀 Full Laya GPU Fine-Tuning & Dual Benchmark Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pchavali09/laya-case-routing-poc/blob/review/next-round/notebooks/laya_gpu_full_finetune.ipynb)

### Overview & Protocol
- **Source:** Official `laya.train.finetune` API (ConvAI Innovations / NandhaKishorM).
- **Laya Version:** Pinned to `laya==0.4.13` (matching local environment).
- **Training Data (Option A):** Trains on `data/einstein_hard_benchmark/train_closed.csv` (800 closed cases, 40 per theme across 20 themes). Neither test set is ever seen during training or validation.
- **Validation Split:** 1 theme per department (200 cases total) held out strictly by theme for validation and temperature calibration.
- **Batching Architecture:** `micro_batch=4, grad_accum=16` (effective batch size 64 across optimizer updates).
- **Encoder Unfreezing:** `freeze_encoder=False` (full backpropagation through the Transformer encoder: ModernBERT-large 421M params).
- **Multi-Seed Stability:** Runs across 3 seeds (`2026`, `2027`, `2028`). Saves each checkpoint, reloads it via canonical `laya.load()`, verifies weight divergence from base model, and scores both test sets.
- **Integrity Assertions:** Weight divergence assertion vs base model, text-level leakage checks (zero identical Subject or Description strings), and prediction validity assertions.

### Quick Run Guide for Pavan
1. Go to **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.
2. Click **Runtime -> Run all**.
3. When finished, all prediction CSVs and `colab_training_summary.json` will be packaged into `laya_finetune_results.zip` and downloaded to your computer.

In [ ]:
# Step 1: Environment & Hardware Verification (Fail loudly if no GPU)
import os, sys, time, json, math, hashlib, shutil, importlib.metadata
import torch

if not torch.cuda.is_available():
    raise SystemError("FATAL: No CUDA GPU detected! Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"CUDA Device: {gpu_name}")
print(f"VRAM Available: {gpu_vram_gb:.2f} GB")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# Step 2: Install Pinned Dependencies (Exact local version 0.4.13)
!pip install -q "laya==0.4.13" "transformers>=4.40.0" "torch>=2.2.0" "safetensors>=0.4.0" "scikit-learn" "pandas" "numpy" "tabulate"

In [ ]:
# Step 3: Self-Contained Dataset Unpacker (Option A: Hard train set unified)
import io, base64, zipfile
import pandas as pd

DATA_ZIP_B64 = "..."

zip_bytes = base64.b64decode(DATA_ZIP_B64)
with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as z:
    z.extractall(".")

ORIG_DIR = "data/einstein_benchmark"
HARD_DIR = "data/einstein_hard_benchmark"

# Option A: unified training set is data/einstein_hard_benchmark/train_closed.csv everywhere
train_df = pd.read_csv(f"{HARD_DIR}/train_closed.csv")
orig_test_df = pd.merge(pd.read_csv(f"{ORIG_DIR}/test_blind.csv"), pd.read_csv(f"{ORIG_DIR}/test_truth.csv"), on="ExternalId__c")
hard_test_df = pd.merge(pd.read_csv(f"{HARD_DIR}/test_blind.csv"), pd.read_csv(f"{HARD_DIR}/test_truth.csv"), on="ExternalId__c")

print(f"Loaded {len(train_df)} training cases ({HARD_DIR}/train_closed.csv).")
print(f"Loaded {len(orig_test_df)} original blind test cases.")
print(f"Loaded {len(hard_test_df)} group-held-out hard test cases.")

In [ ]:
# Step 4: Build Group-Held-Out Validation Split FROM Training Cases (No test leakage)
# 1 theme per department held out (200 cases validation, 600 cases train)
VAL_THEMES = [
    "license_seat_proration_and_credits",  # Billing
    "container_and_pod_crashes",            # Technical
    "annual_contract_renewals",            # Sales
    "office_moves_and_desk_reconfiguration",# Operations
    "recruiting_and_employment_verification"# Other
]

val_mask = train_df["Theme__c"].isin(VAL_THEMES)
train_split = train_df[~val_mask].copy()
val_split = train_df[val_mask].copy()

print(f"Training Split: {len(train_split)} cases across {train_split['Theme__c'].nunique()} themes.")
print(f"Validation Split: {len(val_split)} cases across {val_split['Theme__c'].nunique()} themes.")

# Format for Laya
for df in [train_split, val_split]:
    df["text"] = "Subject: " + df["Subject"].astype(str) + "\nDescription: " + df["Description"].astype(str)
    df["label"] = df["Department__c"]

train_split[["text", "label"]].to_csv("train_split.csv", index=False)
val_split[["text", "label"]].to_csv("val_split.csv", index=False)
print("Training files prepared: train_split.csv and val_split.csv.")

# Text-Level Leakage Assertions: Zero identical Subject or Description strings
train_val_subjects = set(train_df["Subject"].astype(str).str.strip())
train_val_descriptions = set(train_df["Description"].astype(str).str.strip())

for test_name, tdf in [("Original Test Set", orig_test_df), ("Hard Test Set", hard_test_df)]:
    test_subjs = set(tdf["Subject"].astype(str).str.strip())
    test_descs = set(tdf["Description"].astype(str).str.strip())
    subj_leak = train_val_subjects.intersection(test_subjs)
    desc_leak = train_val_descriptions.intersection(test_descs)
    assert len(subj_leak) == 0, f"Integrity Failure: Subject leakage found in {test_name}: {subj_leak}"
    assert len(desc_leak) == 0, f"Integrity Failure: Description leakage found in {test_name}: {desc_leak}"

print("✅ Text-Level Leakage Assertions Passed: Zero identical Subject or Description strings between train/val and either test set.")

In [ ]:
# Step 5: Multi-Seed Full Fine-Tuning on GPU (freeze_encoder=False)
import laya
from laya.train import finetune, TrainConfig
import numpy as np
import importlib.metadata

SEEDS = [2026, 2027, 2028]
DEPARTMENTS = ["Billing", "Technical", "Sales", "Operations", "Other"]

def wilson_interval(k, n, z=1.959963984540054):
    if n == 0: return 0.0, 0.0, 0.0
    p = k / n
    denom = 1 + (z**2) / n
    center = (p + (z**2) / (2 * n)) / denom
    spread = z * math.sqrt((p * (1 - p) + (z**2) / (4 * n)) / n) / denom
    return p, max(0.0, center - spread), min(1.0, center + spread)

def compute_ece(confs, accs, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        in_bin = (confs >= bin_boundaries[i]) & (confs < bin_boundaries[i+1] if i < n_bins - 1 else confs <= bin_boundaries[i+1])
        if np.sum(in_bin) > 0:
            ece += (np.sum(in_bin) / len(confs)) * np.abs(np.mean(accs[in_bin]) - np.mean(confs[in_bin]))
    return float(ece)

def evaluate_reloaded_agent(agent, test_df, target_routed_count=173):
    # Question schema matches exact training schema (same instructions and department choices)
    questions = {
        "department": {
            "type": "choice",
            "instructions": "Which department should handle this customer support case?",
            "criteria": DEPARTMENTS
        }
    }
    preds = []
    for _, row in test_df.iterrows():
        state = f"Subject: {row['Subject']}\nDescription: {row['Description']}"
        t0 = time.time()
        res = agent.predict(state, questions)
        case_lat = (time.time() - t0) * 1000.0
        ans = res["answers"]["department"]
        pred_choice = ans["choice"]
        conf = float(ans.get("confidence", ans.get("answer_confidence", 0.0)))
        preds.append({
            "ExternalId__c": row["ExternalId__c"],
            "true_department": row["true_department"],
            "predicted_department": pred_choice,
            "confidence": conf,
            "latency_ms": round(case_lat, 2),
            "is_correct": bool(pred_choice == row["true_department"])
        })
    df_preds = pd.DataFrame(preds)
    
    # Integrity Assertions: verify predictions came from reloaded agent
    assert len(df_preds) == len(test_df), "Incomplete predictions!"
    assert df_preds["confidence"].min() >= 0.0 and df_preds["confidence"].max() <= 1.0, "Invalid confidence values!"
    assert set(df_preds["predicted_department"]).issubset(set(DEPARTMENTS)), "Invalid department predictions!"
    
    tot = len(df_preds)
    corr = int(df_preds["is_correct"].sum())
    p_ovr, l_ovr, u_ovr = wilson_interval(corr, tot)
    
    sorted_df = df_preds.sort_values(by="confidence", ascending=False).reset_index(drop=True)
    routed_df = sorted_df.iloc[:target_routed_count]
    rout_corr = int(routed_df["is_correct"].sum())
    p_rout, l_rout, u_rout = wilson_interval(rout_corr, len(routed_df))
    ece = compute_ece(df_preds["confidence"].values, df_preds["is_correct"].values.astype(float))
    
    return {
        "overall_correct": corr,
        "overall_accuracy": p_ovr,
        "overall_ci": [l_ovr, u_ovr],
        "cases_routed": len(routed_df),
        "routed_correct": rout_corr,
        "routed_accuracy": p_rout,
        "routed_ci": [l_rout, u_rout],
        "ece": round(ece, 4)
    }, df_preds

# Load base model to snapshot reference weights for divergence assertion, then free GPU memory
base_agent = laya.load("convaiinnovations/laya")
base_weights = {k: v.cpu().clone() for k, v in base_agent.model.state_dict().items()}
del base_agent
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Base model reference snapshot captured and GPU memory freed.")

all_seed_results = []
for seed in SEEDS:
    print(f"\n{'='*70}")
    print(f"RUNNING FINE-TUNE SEED: {seed}")
    print(f"{'='*70}")
    
    out_dir = f"./checkpoint_seed_{seed}"
    os.makedirs(out_dir, exist_ok=True)
    
    config = TrainConfig(
        epochs=4,
        micro_batch=4,
        grad_accum=16,
        encoder_lr=2.5e-5,
        head_lr=1e-4,
        freeze_encoder=False,  # FULL TRANSFORMER BACKBONE FINE-TUNE
        seed=seed,
        calib_seed=seed,
        text_column="text",
        label_column="label",
        question_id="department",
        instructions="Which department should handle this customer support case?",
        eval_data="val_split.csv"
    )
    
    t0 = time.time()
    summary = finetune(
        data="train_split.csv",
        model_dir="convaiinnovations/laya",
        output_dir=out_dir,
        config=config,
        device="cuda"
    )
    t_elapsed = time.time() - t0
    print(f"Seed {seed} finished in {t_elapsed:.1f}s ({t_elapsed/60:.2f} min).")
    
    # Reload saved checkpoint with canonical laya.load(out_dir)
    reloaded_agent = laya.load(out_dir)
    
    # Integrity Assertion: Weight divergence check
    reloaded_weights = {k: v.cpu().clone() for k, v in reloaded_agent.model.state_dict().items()}
    diff_tensors = [k for k in base_weights if not torch.equal(base_weights[k], reloaded_weights[k])]
    assert len(diff_tensors) > 0, f"Integrity Failure: Reloaded weights are IDENTICAL to base model for seed {seed}!"
    print(f"✅ Integrity Assertion Passed: {len(diff_tensors)} weight tensors differ between base and reloaded model.")
    
    # Evaluate on both test sets using matching question schema
    orig_res, orig_preds = evaluate_reloaded_agent(reloaded_agent, orig_test_df)
    hard_res, hard_preds = evaluate_reloaded_agent(reloaded_agent, hard_test_df)
    
    orig_preds.to_csv(f"predictions_orig_seed_{seed}.csv", index=False)
    hard_preds.to_csv(f"predictions_hard_seed_{seed}.csv", index=False)
    
    all_seed_results.append({
        "seed": seed,
        "training_time_s": round(t_elapsed, 2),
        "differing_weight_tensors": len(diff_tensors),
        "original_set": orig_res,
        "hard_set": hard_res
    })

# Export summary JSON (strict package metadata extraction)
laya_version = importlib.metadata.version("laya")
final_summary = {
    "gpu_device": gpu_name,
    "gpu_vram_gb": round(gpu_vram_gb, 2),
    "laya_version": laya_version,
    "training_data": f"{HARD_DIR}/train_closed.csv",
    "micro_batch": 4,
    "grad_accum": 16,
    "seeds": SEEDS,
    "freeze_encoder": False,
    "runs": all_seed_results
}
with open("colab_training_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)

print("\nAll 3 seeds completed successfully! Saved colab_training_summary.json")

In [ ]:
# Step 6: Package and Download Outputs
!zip -q laya_finetune_results.zip predictions_orig_seed_*.csv predictions_hard_seed_*.csv colab_training_summary.json

try:
    from google.colab import files
    files.download("laya_finetune_results.zip")
    print("Triggered download of laya_finetune_results.zip.")
except Exception as e:
    print("Results saved locally to laya_finetune_results.zip. Download manually from file browser.")